In [ ]:
# Cell 1: Setup and Installations
# This first cell sets up the environment by installing necessary libraries and cloning the required repositories.
# We will use PyTorch and torchvision, which are a great combination for object detection.
# MattNet is a research model, so we'll clone a public implementation from GitHub.

!pip install torch torchvision
!git clone https://github.com/lichengunc/MAttNet.git
# Note: You may need to compile some custom C++/CUDA extensions depending on the MattNet implementation.
# This can be done by navigating into the lib directory of the cloned repository and running:
# cd MAttNet/lib
# python setup.py build develop

# Import all necessary libraries for the rest of the notebook.
import os
import json
import torch
import torchvision
from PIL import Image
from tqdm import tqdm
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Define the device to use (GPU if available).
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device: {device}")



In [ ]:
import os
import json

KAGGLE_DATA_ROOT = "/kaggle/input/visualgenome"
print(f"Using Kaggle data root: {KAGGLE_DATA_ROOT}")

region_graphs_path = None
image_data_path = None
IMAGE_DIR_1 = None
IMAGE_DIR_2 = None

print("Searching for data files and image directories within the downloaded dataset...")

for root, dirs, files in os.walk(KAGGLE_DATA_ROOT):
    if 'region_graphs.json' in files and region_graphs_path is None:
        region_graphs_path = os.path.join(root, 'region_graphs.json')
    
    if 'image_data.json' in files and image_data_path is None:
        image_data_path = os.path.join(root, 'image_data.json')

    if 'VG_100K' in dirs and IMAGE_DIR_1 is None:
        IMAGE_DIR_1 = os.path.join(root, 'VG_100K')
    if 'VG_100K_2' in dirs and IMAGE_DIR_2 is None:
        IMAGE_DIR_2 = os.path.join(root, 'VG_100K_2')

    if region_graphs_path and image_data_path and IMAGE_DIR_1 and IMAGE_DIR_2:
        break

if not region_graphs_path:
    raise FileNotFoundError("Could not locate the file 'region_graphs.json'.")
if not image_data_path:
    raise FileNotFoundError("Could not locate the file 'image_data.json'.")
if not IMAGE_DIR_1:
    raise FileNotFoundError("Could not locate the directory 'VG_100K'.")
if not IMAGE_DIR_2:
    raise FileNotFoundError("Could not locate the directory 'VG_100K_2'.")

print(f"region_graphs.json found at: {region_graphs_path}")
print(f"image_data.json found at: {image_data_path}")
print(f"Image directory 1 found in: {IMAGE_DIR_1}")
print(f"Image directory 2 found in: {IMAGE_DIR_2}")

with open(region_graphs_path, 'r') as f:
    region_graphs = json.load(f)

with open(image_data_path, 'r') as f:
    image_data = json.load(f)


print(f"Loaded {len(region_graphs)} region graph entries.")
print(f"Loaded {len(image_data)} image data entries.")


image_id_to_path = {}
for img in image_data:
    image_id = img['image_id']
    path1 = os.path.join(IMAGE_DIR_1, f"{image_id}.jpg")
    path2 = os.path.join(IMAGE_DIR_2, f"{image_id}.jpg")
    if os.path.exists(path1):
        image_id_to_path[image_id] = path1
    elif os.path.exists(path2):
        image_id_to_path[image_id] = path2


subset_size = 50000
image_ids = list(image_id_to_path.keys())[:subset_size]

if not image_ids:
    raise ValueError("No images were found in the dataset. Please check the dataset structure.")
print(f"Using a subset of {len(image_ids)} images for training.")


image_annotations = {}
for entry in region_graphs:
    image_id = entry['image_id']
    if image_id in image_id_to_path:
        image_annotations[image_id] = entry['regions']



In [ ]:

class VisualGenomeDataset(Dataset):
    def __init__(self, image_ids, image_annotations, image_id_to_path, transforms=None):
        self.image_ids = image_ids
        self.image_annotations = image_annotations
        self.image_id_to_path = image_id_to_path
        self.transforms = transforms

    def __getitem__(self, idx):
       
        try:
            image_id = self.image_ids[idx]
            
            img_path = self.image_id_to_path[image_id]
            img = Image.open(img_path).convert("RGB")
            
            regions = self.image_annotations.get(image_id, [])

            boxes = []
            labels = []
            
            for region in regions:
                x, y, w, h = region['x'], region['y'], region['width'], region['height']
                boxes.append([x, y, x + w, y + h])
                labels.append(1) 
            
            if not boxes:
                boxes = torch.zeros((0, 4), dtype=torch.float32)
                labels = torch.zeros(0, dtype=torch.int64)
            else:
                boxes = torch.as_tensor(boxes, dtype=torch.float32)
                labels = torch.as_tensor(labels, dtype=torch.int64)

            target = {}
            target["boxes"] = boxes
            target["labels"] = labels
            target["image_id"] = torch.tensor([image_id])

            if self.transforms is not None:
                img = self.transforms(img)

            return img, target
        
        except Exception as e:
            print(f"Error processing image ID {self.image_ids[idx]}: {e}")
            return (torch.zeros((3, 224, 224), dtype=torch.float32), 
                    {'boxes': torch.zeros((0, 4), dtype=torch.float32), 'labels': torch.zeros(0, dtype=torch.int64)})


    def __len__(self):
        return len(self.image_ids)


def get_transform():
    return torchvision.transforms.ToTensor()

if 'image_ids' in locals() and 'image_annotations' in locals() and 'image_id_to_path' in locals():
    dataset = VisualGenomeDataset(image_ids, image_annotations, image_id_to_path, transforms=get_transform())

    def collate_fn(batch):
        batch = [item for item in batch if item is not None]
        if not batch:
            return None, None
        return tuple(zip(*batch))

    data_loader = DataLoader(
        dataset,
        batch_size=2,  
        shuffle=True,
        num_workers=0,
        collate_fn=collate_fn
    )
else:
    print("Skipping Dataset and DataLoader creation due to missing data.")



In [ ]:

num_classes = 2

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)


model.to(device)


In [ ]:

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    
    progress_bar = tqdm(data_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for images, targets in progress_bar:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        progress_bar.set_postfix(loss=losses.item())
        
    if lr_scheduler is not None:
        lr_scheduler.step()
        
    print(f"Epoch {epoch+1} finished. Total Loss: {losses.item()}")

torch.save(model.state_dict(), 'faster_rcnn_visual_genome.pth')
print("Training complete. Model saved.")



In [ ]:

output_path = 'faster_rcnn_visual_genome_final.pth'


torch.save(model.state_dict(), output_path)
print(f"Model saved successfully to: {os.path.abspath(output_path)}")

torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, 'checkpoint_visual_genome.pth')
print("Checkpoint (model + optimizer) saved.")

In [ ]:
import random
import numpy as np

def visualize_inference(model, dataset, idx=None, threshold=0.5):
    """
    Runs inference on a single image and plots the results.
    Args:
        model: The trained Faster R-CNN model.
        dataset: The validation/test dataset.
        idx: Index of the image to test. If None, picks random.
        threshold: Confidence score threshold to show boxes.
    """
    model.eval()
    

    if idx is None:
        idx = random.randint(0, len(dataset) - 1)
    
    img_tensor, _ = dataset[idx]
    

    with torch.no_grad():

        prediction = model([img_tensor.to(device)])[0]

    boxes = prediction['boxes'].cpu().numpy()
    scores = prediction['scores'].cpu().numpy()
    

    qualified_indices = np.where(scores > threshold)[0]
    qualified_boxes = boxes[qualified_indices]
    qualified_scores = scores[qualified_indices]
    
    print(f"Image Index: {idx}")
    print(f"Total detections: {len(boxes)}")
    print(f"Detections > {threshold} score: {len(qualified_boxes)}")


    img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
    

    fig, ax = plt.subplots(1, figsize=(12, 9))
    ax.imshow(img_np)
    
    for i, box in enumerate(qualified_boxes):
        x_min, y_min, x_max, y_max = box
        

        width = x_max - x_min
        height = y_max - y_min
        

        rect = patches.Rectangle(
            (x_min, y_min), 
            width, 
            height, 
            linewidth=2, 
            edgecolor='r', 
            facecolor='none'
        )
        

        ax.add_patch(rect)
        

        ax.text(
            x_min, 
            y_min - 5, 
            f'{qualified_scores[i]:.2f}', 
            color='white', 
            fontsize=10, 
            bbox=dict(facecolor='red', alpha=0.5, pad=0)
        )

    plt.title(f"Inference Result (Threshold: {threshold})")
    plt.axis('off')
    plt.show()


visualize_inference(model, dataset, threshold=0.6)